# Histograms (cont.), Functions

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


## Histogram: Density

In [ ]:
# datascience
# Data from https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/
top_movies = Table.read_table('top_movies_2026.csv')
ages = 2026 - top_movies.column('Year')
top_movies = top_movies.with_column('Age', ages)
top_movies


In [ ]:
# polars
# Data from https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/
top_movies_pl = pl.read_csv('top_movies_2026.csv')
ages_pl = 2026 - top_movies_pl.get_column('Year')
top_movies_pl = top_movies_pl.with_columns(pl.Series('Age', ages_pl))
top_movies_pl


In [ ]:
# datascience
my_bins = make_array(0, 5, 10, 15, 25, 60)
binned_data = top_movies.bin('Age', bins = my_bins)
binned_data


In [ ]:
# polars
my_bins_pl = pl.Series('values', [0, 5, 10, 15, 25, 60])
binned_data_pl = top_movies_pl.get_column('Age').hist(bins=(my_bins_pl).to_list()).rename({'breakpoint': 'bin', 'count': 'Age count'}).drop('category')
binned_data_pl


In [ ]:
# datascience
# what does normed=True, normed=False do?
top_movies.hist('Age', bins = my_bins, normed=True)


In [ ]:
# polars
# hvplot
# what does normed=True, normed=False do?
top_movies_pl.hvplot.hist(y='Age', normed=True, bins=(my_bins_pl).to_list())


### **Discussion Question**: Compare the bins $[0, 5)$ and $[15, 25)$. 

- Which one has more movies?
- Which one is more crowded?

---

### Flat Tops and the Level of Detail (7.2.5)

In [ ]:
# datascience
uneven = make_array(600, 650, 700, 800, 1000, 3000)
top_movies = top_movies.with_column('Worldwide',
                       top_movies.column('Worldwide Lifetime Gross') / 1e6
                      )
top_movies.hist('Worldwide', bins=uneven, unit='Million Dollars')


In [ ]:
# polars
# hvplot
uneven_pl = pl.Series('values', [600, 650, 700, 800, 1000, 3000])
top_movies_pl = top_movies_pl.with_columns((top_movies_pl.get_column('Worldwide Lifetime Gross') / 1e6).alias('Worldwide'))
top_movies_pl.hvplot.hist(y='Worldwide', normed=True, bins=(uneven_pl).to_list(), xlabel='Worldwide (Million Dollars)')


In [ ]:
# datascience
top_movies.bin('Worldwide', bins=uneven)


In [ ]:
# polars
top_movies_pl.get_column('Worldwide').hist(bins=(uneven_pl).to_list()).rename({'breakpoint': 'bin', 'count': 'Worldwide count'}).drop('category')


In [ ]:
# datascience
some_tiny_bins = make_array(600, 650, 700, 800, 820, 840, 860, 880, 900, 920, 940, 960, 980, 1000, 3000)
top_movies.hist('Worldwide', bins=some_tiny_bins, unit='Million Dollars')


In [ ]:
# polars
# hvplot
some_tiny_bins_pl = pl.Series('values', [600, 650, 700, 800, 820, 840, 860, 880, 900, 920, 940, 960, 980, 1000, 3000])
top_movies_pl.hvplot.hist(y='Worldwide', normed=True, bins=(some_tiny_bins_pl).to_list(), xlabel='Worldwide (Million Dollars)')


In [ ]:
# datascience
top_movies.hist('Worldwide', bins=20, unit='Million Dollars')


In [ ]:
# polars
# hvplot
top_movies_pl.hvplot.hist(y='Worldwide', normed=True, bins=20, xlabel='Worldwide (Million Dollars)')


## Defining Functions ##  

Example: Create a function that takes a numerical input and triples it.

In [ ]:
def triple(x):
    return 3 * x

In [ ]:
triple(3)

In [ ]:
triple(triple(3))

We can also assign a value to a name, and call the function on the name:

In [ ]:
num = 4

In [ ]:
triple(num)

In [ ]:
triple(num * 5)

### Functions are Type-Agnostic

In [ ]:
triple('ha')

In [ ]:
# datascience
np.arange(4)


In [ ]:
# polars
pl.int_range(4, eager=True)


Feed the array above into our function `triple` to see what is produced:

In [ ]:
# datascience
triple(np.arange(4))


In [ ]:
# polars
triple(pl.int_range(4, eager=True))


---

### Discussion ###

- What does the following function do?
- What type of input does it take?
- What type of output does it produce?
- What's a good name for the function?

```python
def f(s):     
      return np.round(s / sum(s) * 100, 2)
```

<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>

In [ ]:
# datascience
def percent_of_total(s):
    return np.round(s / sum(s) * 100, 2)


In [ ]:
# polars
def percent_of_total_pl(s):
    return (s / sum(s) * 100).round(2)


In [ ]:
# datascience
first_four=make_array(1,2,3,4)
first_four


In [ ]:
# polars
first_four_pl=pl.Series('values', [1,2,3,4])
first_four_pl


In [ ]:
percent_of_total(first_four)

In [ ]:
# datascience
percent_of_total(make_array(1, 213, 38))


In [ ]:
# polars
percent_of_total_pl(pl.Series('values', [1, 213, 38]))


---

## A strategy for defining functions:
- Give generic, descriptive names to some example inputs.
- Try writing the code that gives you the right output for that example.
- Once that works, put that code into a `def` statement (without the example; with a `return`).
- Try calling it on some different example inputs (to make sure it's flexible).

---

## Functions Can Take Multiple Arguments

### Example: Calculate the Hypotenuse Length of a Right Triangle


Pythagoras's Theorem: If $x$ and $y$ denote the lengths of the right-angle sides, then the hypotenuse length $h$ satisfies:

$$ h^2 = x^2 + y^2 \qquad \text{which implies}\qquad \hspace{20 pt} h = \sqrt{ x^2 + y^2 } $$

In [ ]:
def hypotenuse(x, y):
    hypot_squared = (x ** 2 + y ** 2)
    hypot = hypot_squared ** 0.5
    return hypot

In [ ]:
hypotenuse(1, 2)

In [ ]:
hypotenuse(3, 4)

We could've typed the body all in one line. Do you find this more readable or less readable than the original version?

In [ ]:
def hypotenuse(x,y):
    return (x ** 2 + y ** 2) ** 0.5

In [ ]:
hypotenuse(9, 12)

### Example: A function that takes the year of birth of a person and produces their age in years. ###

In [ ]:
def age(year):
    age = 2026 - year
    return age

In [ ]:
age(1980)

Now add some bells and whistles:  Take person's name and year of birth (two arguments). Produce a sentence that states how old they are.

In [ ]:
def name_and_age(name, year):
    return name + ' is ' + str(age(year)) + ' years old.'

In [ ]:
name_and_age('Lisa', 1991)

---

## Apply

Characters from [_The Office_](https://en.wikipedia.org/wiki/The_Office_(American_TV_series))

In [ ]:
# datascience
ages = Table().with_columns(
    'Person', make_array('Jim', 'Pam', 'Michael', 'Creed'),
    'Birth Year', make_array(1985, 1988, 1967, 1904)
)
ages


In [ ]:
# polars
ages_pl = pl.DataFrame([
    (pl.Series('values', ['Jim', 'Pam', 'Michael', 'Creed'])).alias('Person'),
    (pl.Series('values', [1985, 1988, 1967, 1904])).alias('Birth Year')
])
ages_pl


In [ ]:
# datascience
ages.apply(age, 'Birth Year')


In [ ]:
# polars
ages_pl.get_column('Birth Year').map_elements(age, return_dtype=pl.Int64)


In [ ]:
# datascience

make_array(age(ages.column('Birth Year').item(0)),
           age(ages.column('Birth Year').item(1)),
           age(ages.column('Birth Year').item(2)),
           age(ages.column('Birth Year').item(3)))


In [ ]:
# polars

pl.Series('values', [age(ages_pl.get_column('Birth Year').item(0)),
           age(ages_pl.get_column('Birth Year').item(1)),
           age(ages_pl.get_column('Birth Year').item(2)),
           age(ages_pl.get_column('Birth Year').item(3))])


In [ ]:
# datascience
ages.apply(name_and_age, 'Person', 'Birth Year')


In [ ]:
# polars
ages_pl.select(pl.struct(['Person', 'Birth Year']).map_elements(lambda row: name_and_age(row['Person'], row['Birth Year']), return_dtype=pl.String)).to_series()
